In [1]:
# 1) Imports and data loading

import pandas as pd
import numpy as np

df = pd.read_csv("data/covid19_patient_symptoms_diagnosis.csv")
rows, columns = df.shape
print(f"COVID-19 dataset loaded successfully: {rows} rows and {columns} columns.")

In [2]:
# 2) Initial data inspection

print("--- First 5 Rows of the Dataset ---")
print(df.head())
print("\n--- Last 5 Rows of the Dataset ---")
print(df.tail())
print("\n--- Column Data Types ---")
print(df.dtypes)
print("\n--- Dataset Info (Null Counts & Types) ---")
df.info()

In [3]:
# 3) Basic demographic summary

total_records = len(df)
print("Total Records in Dataset:", total_records)

gender_summary = df['gender'].value_counts()
print("\n--- Gender Breakdown ---")
print(gender_summary)

mean_age = df['age'].mean()
print(f"\nMean Age of Dataset: {mean_age:.2f} (Rounded: {round(mean_age)})")
print("\n--- Age Descriptive Statistics ---")
print(df['age'].describe())

In [4]:
# 4) Data cleaning — duplicates and missing values

print("Total Duplicate Rows:", df.duplicated().sum())

print("\n--- Missing Value Count per Column ---")
print(df.isnull().sum())

print("\nUnique values in comorbidity column:", df['comorbidity'].unique())
counts = df['comorbidity'].value_counts(dropna=False)
percent = df['comorbidity'].value_counts(normalize=True, dropna=False) * 100
print("\n--- Comorbidity Distribution (Before Cleaning) ---")
print(pd.DataFrame({'Count': counts, 'Percent': percent.round(2)}))

df_cleaned = df.copy()
df_cleaned['comorbidity'] = df_cleaned['comorbidity'].fillna("None")

print(f"\nMissing values BEFORE cleaning: {df['comorbidity'].isnull().sum()}")
print(f"Missing values AFTER cleaning:  {df_cleaned['comorbidity'].isnull().sum()}")

counts_clean = df_cleaned['comorbidity'].value_counts(dropna=False)
percent_clean = df_cleaned['comorbidity'].value_counts(normalize=True, dropna=False) * 100
print("\n--- Comorbidity Distribution (After Cleaning) ---")
print(pd.DataFrame({'Count': counts_clean, 'Percent': percent_clean.round(2)}))

In [5]:
# 5) Outlier detection for age using IQR method

print("--- Age Descriptive Statistics ---")
print(df_cleaned['age'].describe(), '\n')

Q1 = df_cleaned['age'].quantile(0.25)
Q3 = df_cleaned['age'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_cleaned[(df_cleaned['age'] < lower_bound) | (df_cleaned['age'] > upper_bound)]

print('-' * 30)
print(" AGE OUTLIER ANALYSIS ")
print('-' * 30)
print(f"Quartile 1 (25%): {Q1:>10}")
print(f"Quartile 3 (75%): {Q3:>10}")
print(f"IQR:              {IQR:>10}")
print("-" * 30)
print(f"Lower Bound:      {lower_bound:>10}")
print(f"Upper Bound:      {upper_bound:>10}")
print("-" * 30)
print(f"Outliers Found:   {len(outliers):>10}")
print("-" * 30)

In [6]:
# 6) Age group aggregation into 6 life stage categories

bins = [0, 18, 35, 50, 60, 80, 100]
labels = ['Minors (0-17)', 'Young Adult (18-34)', 'Adult (35-49)',
          'Middle Aged (50-64)', 'Senior (65-79)', 'Elderly (80+)']

df_cleaned['age_categories'] = pd.cut(df_cleaned['age'], bins=bins, labels=labels, right=False)

category_counts = df_cleaned['age_categories'].value_counts().sort_index()
total_patients = category_counts.sum()

print("--- Patient Count per Age Category ---")
print(category_counts)
print(f"\nTotal Patients Across All Groups: {total_patients}")

In [7]:
# 7) Age and gender breakdown by group

print("--- Age Statistics by Gender ---")
print(df_cleaned.groupby('gender')['age'].agg(['mean', 'median', 'min', 'max', 'count']))

gender_within_age = pd.crosstab(df_cleaned['age_categories'], df_cleaned['gender'], normalize='index') * 100
age_within_gender = pd.crosstab(df_cleaned['age_categories'], df_cleaned['gender'], normalize='columns') * 100

print("\n--- Gender Split WITHIN Each Age Category (%) ---")
print(gender_within_age.map('{:.2f}%'.format))

print("\n--- Age Distribution WITHIN Each Gender (%) ---")
print(age_within_gender.map('{:.2f}%'.format))

In [8]:
# 8) Overall symptom frequency and rate by COVID result

symptoms = ['fever', 'dry_cough', 'sore_throat', 'fatigue', 'headache',
            'shortness_of_breath', 'loss_of_smell', 'loss_of_taste', 'chest_pain']

symptom_counts = df_cleaned[symptoms].sum().sort_values(ascending=False)
symptom_pct = (symptom_counts / len(df_cleaned) * 100).round(2)

print("--- Overall Symptom Frequency ---")
print(pd.DataFrame({'Count': symptom_counts, 'Percent (%)': symptom_pct}))

symptom_by_result = df_cleaned.groupby('covid_result')[symptoms].mean() * 100
symptom_by_result.index = ['Negative', 'Positive']
print("\n--- Symptom Rate by COVID Result (%) ---")
print(symptom_by_result.T.round(2))

In [9]:
# 9) Symptom analysis broken down by age group

symptom_by_age = df_cleaned.groupby('age_categories', observed=False)[symptoms].mean() * 100

print("--- Symptom Rate by Age Group (%) ---")
print(symptom_by_age.round(2))

print("\n--- Most Common Symptom Per Age Group ---")
print(symptom_by_age.idxmax(axis=1))

print("\n--- Which Age Group Has the Highest Rate of Each Symptom ---")
for symptom in symptoms:
    top_group = symptom_by_age[symptom].idxmax()
    top_pct = symptom_by_age[symptom].max().round(2)
    print(f"{symptom:<25} → {top_group} ({top_pct}%)")

In [10]:
# 10) COVID positive rate and symptom burden by age group

df_cleaned['symptom_count'] = df_cleaned[symptoms].sum(axis=1)

print("--- Avg Symptom Count & COVID Positive Rate by Age Group ---")
age_summary = df_cleaned.groupby('age_categories', observed=False).agg(
    Avg_Symptoms=('symptom_count', 'mean'),
    Positive_Rate=('covid_result', 'mean'),
    Patient_Count=('patient_id', 'count')
)
age_summary['Positive_Rate'] = (age_summary['Positive_Rate'] * 100).round(2)
age_summary['Avg_Symptoms'] = age_summary['Avg_Symptoms'].round(2)
print(age_summary)

In [11]:
# 11) Gender demographic summary

gender_demo = df_cleaned.groupby('gender').agg(
    Patient_Count=('patient_id', 'count'),
    Avg_Age=('age', 'mean'),
    Min_Age=('age', 'min'),
    Max_Age=('age', 'max'),
    Positive_Rate=('covid_result', 'mean')
)
gender_demo['Avg_Age'] = gender_demo['Avg_Age'].round(2)
gender_demo['Positive_Rate'] = (gender_demo['Positive_Rate'] * 100).round(2)

print("--- Gender Demographic Summary ---")
print(gender_demo)

In [12]:
# 12) Symptom frequency broken down by gender

symptom_by_gender = df_cleaned.groupby('gender')[symptoms].mean() * 100

print("--- Symptom Rate by Gender (%) ---")
print(symptom_by_gender.T.round(2))

print("\n--- Most Common Symptom Per Gender ---")
print(symptom_by_gender.idxmax(axis=1))

In [13]:
# 13) COVID positive rate and symptom burden by gender

gender_summary = df_cleaned.groupby('gender').agg(
    Avg_Symptoms=('symptom_count', 'mean'),
    Positive_Rate=('covid_result', 'mean'),
    Patient_Count=('patient_id', 'count')
)
gender_summary['Positive_Rate'] = (gender_summary['Positive_Rate'] * 100).round(2)
gender_summary['Avg_Symptoms'] = gender_summary['Avg_Symptoms'].round(2)

print("--- Avg Symptom Count & COVID Positive Rate by Gender ---")
print(gender_summary)

In [14]:
# 14) Symptom dominance by gender showing which gender reports each symptom more

print("--- Which Gender Has the Highest Rate of Each Symptom ---")
for symptom in symptoms:
    top_gender = symptom_by_gender[symptom].idxmax()
    top_pct = symptom_by_gender[symptom].max().round(2)
    diff = (symptom_by_gender[symptom].max() - symptom_by_gender[symptom].min()).round(2)
    print(f"{symptom:<25} → {top_gender} ({top_pct}%)  |  Difference: {diff}%")

In [15]:
# 15) Symptom rate by gender and COVID result combined

print("--- Symptom Rate by Gender & COVID Result (%) ---")
gender_covid_symptoms = df_cleaned.groupby(['gender', 'covid_result'])[symptoms].mean() * 100
gender_covid_symptoms.index = gender_covid_symptoms.index.set_levels(
    ['Negative', 'Positive'], level='covid_result'
)
print(gender_covid_symptoms.round(2).T)

In [16]:
# 16) Comorbidity demographic summary

comorbidity_demo = df_cleaned.groupby('comorbidity').agg(
    Patient_Count=('patient_id', 'count'),
    Avg_Age=('age', 'mean'),
    Min_Age=('age', 'min'),
    Max_Age=('age', 'max'),
    Positive_Rate=('covid_result', 'mean')
)
comorbidity_demo['Avg_Age'] = comorbidity_demo['Avg_Age'].round(2)
comorbidity_demo['Positive_Rate'] = (comorbidity_demo['Positive_Rate'] * 100).round(2)

print("--- Comorbidity Demographic Summary ---")
print(comorbidity_demo)

In [17]:
# 17) Symptom frequency broken down by comorbidity group

symptom_by_comorbidity = df_cleaned.groupby('comorbidity')[symptoms].mean() * 100

print("--- Symptom Rate by Comorbidity Group (%) ---")
print(symptom_by_comorbidity.T.round(2))

print("\n--- Most Common Symptom Per Comorbidity Group ---")
print(symptom_by_comorbidity.idxmax(axis=1))

In [18]:
# 18) COVID positive rate and symptom burden by comorbidity group

comorbidity_summary = df_cleaned.groupby('comorbidity').agg(
    Avg_Symptoms=('symptom_count', 'mean'),
    Positive_Rate=('covid_result', 'mean'),
    Patient_Count=('patient_id', 'count')
)
comorbidity_summary['Positive_Rate'] = (comorbidity_summary['Positive_Rate'] * 100).round(2)
comorbidity_summary['Avg_Symptoms'] = comorbidity_summary['Avg_Symptoms'].round(2)

print("--- Avg Symptom Count & COVID Positive Rate by Comorbidity ---")
print(comorbidity_summary)

In [19]:
# 19) Symptom dominance by comorbidity showing which group reports each symptom most

print("--- Which Comorbidity Group Has the Highest Rate of Each Symptom ---")
for symptom in symptoms:
    top_group = symptom_by_comorbidity[symptom].idxmax()
    top_pct = symptom_by_comorbidity[symptom].max().round(2)
    diff = (symptom_by_comorbidity[symptom].max() - symptom_by_comorbidity[symptom].min()).round(2)
    print(f"{symptom:<25} → {top_group} ({top_pct}%)  |  Difference: {diff}%")

In [20]:
# 20) COVID result demographic summary comparing positive vs negative patients

covid_demo = df_cleaned.groupby('covid_result').agg(
    Patient_Count=('patient_id', 'count'),
    Avg_Age=('age', 'mean'),
    Min_Age=('age', 'min'),
    Max_Age=('age', 'max'),
    Avg_Symptoms=('symptom_count', 'mean'),
    Avg_Oxygen=('oxygen_level', 'mean'),
    Avg_Temp=('body_temperature', 'mean')
)
covid_demo.index = ['Negative', 'Positive']
covid_demo['Avg_Age'] = covid_demo['Avg_Age'].round(2)
covid_demo['Avg_Symptoms'] = covid_demo['Avg_Symptoms'].round(2)
covid_demo['Avg_Oxygen'] = covid_demo['Avg_Oxygen'].round(2)
covid_demo['Avg_Temp'] = covid_demo['Avg_Temp'].round(2)

print("--- COVID Result Demographic Summary ---")
print(covid_demo)

In [21]:
# 21) Symptom frequency and difference between positive and negative patients

symptom_by_covid = df_cleaned.groupby('covid_result')[symptoms].mean() * 100
symptom_by_covid.index = ['Negative', 'Positive']

print("--- Symptom Rate by COVID Result (%) ---")
print(symptom_by_covid.T.round(2))

print("\n--- Symptom Difference: Positive vs Negative (%) ---")
symptom_diff = (symptom_by_covid.loc['Positive'] - symptom_by_covid.loc['Negative']).round(2)
print(symptom_diff.sort_values(ascending=False))

In [22]:
# 22) Strongest symptom predictors of a positive COVID result

print("--- COVID Positive Rate WITH vs WITHOUT Each Symptom ---")
predictor_results = []
for symptom in symptoms:
    with_symptom = df_cleaned[df_cleaned[symptom] == 1]['covid_result'].mean() * 100
    without_symptom = df_cleaned[df_cleaned[symptom] == 0]['covid_result'].mean() * 100
    difference = with_symptom - without_symptom
    predictor_results.append({
        'Symptom': symptom,
        'With (%)': round(with_symptom, 2),
        'Without (%)': round(without_symptom, 2),
        'Difference': round(difference, 2)
    })

print(pd.DataFrame(predictor_results).set_index('Symptom').sort_values('Difference', ascending=False))

In [23]:
# 23) Oxygen level group creation using clinical thresholds

oxy_bins = [0, 85, 90, 95, 101]
oxy_labels = ['Severe (<85)', 'Moderate (85-89)', 'Mild (90-94)', 'Normal (95-100)']

df_cleaned['oxygen_group'] = pd.cut(df_cleaned['oxygen_level'], bins=oxy_bins, labels=oxy_labels, right=False)

oxygen_counts = df_cleaned['oxygen_group'].value_counts().sort_index()
print("--- Patient Count per Oxygen Level Group ---")
print(oxygen_counts)
print(f"\nTotal Patients: {oxygen_counts.sum()}")

In [24]:
# 24) Oxygen level group demographic summary

oxygen_demo = df_cleaned.groupby('oxygen_group', observed=False).agg(
    Patient_Count=('patient_id', 'count'),
    Avg_Age=('age', 'mean'),
    Positive_Rate=('covid_result', 'mean'),
    Avg_Symptoms=('symptom_count', 'mean'),
    Avg_Temp=('body_temperature', 'mean')
)
oxygen_demo['Avg_Age'] = oxygen_demo['Avg_Age'].round(2)
oxygen_demo['Positive_Rate'] = (oxygen_demo['Positive_Rate'] * 100).round(2)
oxygen_demo['Avg_Symptoms'] = oxygen_demo['Avg_Symptoms'].round(2)
oxygen_demo['Avg_Temp'] = oxygen_demo['Avg_Temp'].round(2)

print("--- Oxygen Level Group Demographic Summary ---")
print(oxygen_demo)

In [25]:
# 25) Symptom frequency and dominance broken down by oxygen level group

symptom_by_oxygen = df_cleaned.groupby('oxygen_group', observed=True)[symptoms].mean() * 100

print("--- Symptom Rate by Oxygen Level Group (%) ---")
print(symptom_by_oxygen.round(2))

print("\n--- Most Common Symptom Per Oxygen Level Group ---")
print(symptom_by_oxygen.idxmax(axis=1))

print("\n--- Which Oxygen Group Has the Highest Rate of Each Symptom ---")
for symptom in symptoms:
    top_group = symptom_by_oxygen[symptom].idxmax()
    top_pct = symptom_by_oxygen[symptom].max().round(2)
    diff = (symptom_by_oxygen[symptom].max() - symptom_by_oxygen[symptom].min()).round(2)
    print(f"{symptom:<25} → {top_group} ({top_pct}%)  |  Difference: {diff}%")

In [26]:
# 26) COVID positive rate and symptom burden by oxygen level group

oxygen_summary = df_cleaned.groupby('oxygen_group', observed=True).agg(
    Avg_Symptoms=('symptom_count', 'mean'),
    Positive_Rate=('covid_result', 'mean'),
    Patient_Count=('patient_id', 'count')
)
oxygen_summary['Positive_Rate'] = (oxygen_summary['Positive_Rate'] * 100).round(2)
oxygen_summary['Avg_Symptoms'] = oxygen_summary['Avg_Symptoms'].round(2)

print("--- Avg Symptom Count & COVID Positive Rate by Oxygen Level Group ---")
print(oxygen_summary)